# Despliegue de API OMR en Colab
Este cuaderno lanza el backend completo de Frig.io API Gateway + GPU Workerpara exponerlo mediante Cloudflare.


## 1. Montar Google Drive
Para cargar el modelo entrenado previamente lo cogemos de Google Drive. 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
MODEL_PATH = Path('/content/drive/MyDrive/OMR_Dataset_2/qwen2.5_vl_omr_lora_v2')
if not MODEL_PATH.is_dir():
    raise FileNotFoundError(
        f'No se encontró el modelo fine-tuneado en {MODEL_PATH}. '
        'Ejecuta primero Colab_OMR_Finetune.ipynb o corrige MODEL_PATH.'
    )
print(f'Modelo fine-tuneado encontrado: {MODEL_PATH}')

## 2. Instalar dependencias
Instala Unsloth, FastAPI, uvicorn y Cloudflared.

In [ ]:
!pip install unsloth unsloth_zoo trl peft accelerate bitsandbytes xformers
!pip install fastapi uvicorn python-multipart pdf2image qwen-vl-utils pillow numpy requests linearized-musicxml
!apt-get install -y poppler-utils > /dev/null 2>&1
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb


## 3. Instalar scripts de la API
Se inyectan en Colab los scripts locales de la API (api.py, model_service.py, omr_processor.py).

In [ ]:
%%writefile /content/omr_processor.py
import asyncio
import logging
import re
from pathlib import Path

logger = logging.getLogger(__name__)

SYSTEM_PROMPT_INFERENCE = "Transcribe this musical score to LMX format."

class OMRProcessor:
    FINETUNED_MODEL_PATH = "/content/drive/MyDrive/OMR_Dataset_2/qwen2.5_vl_omr_lora_v2"
    MAX_NEW_TOKENS_SYSTEM = 512

    def __init__(self):
        self.model = None
        self.tokenizer = None
        self._load_model()

    def _load_model(self):
        try:
            from unsloth import FastVisionModel
            import torch
            import gc

            gc.collect()
            torch.cuda.empty_cache()

            model_path = Path(self.FINETUNED_MODEL_PATH)
            if not model_path.is_dir():
                raise FileNotFoundError(
                    f"No se encontro el modelo fine-tuneado en {model_path}. "
                    "Deployment no puede arrancar con el modelo base."
                )
            logger.info(f"Modelo fine-tuneado detectado: {model_path}")

            logger.info("Instanciando en modo ahorro RAM (low_cpu_mem_usage=True)...")
            self.model, self.tokenizer = FastVisionModel.from_pretrained(
                str(model_path),
                load_in_4bit=True,
                use_gradient_checkpointing="unsloth",
                device_map="auto",
                low_cpu_mem_usage=True
            )
            FastVisionModel.for_inference(self.model)
            logger.info("Modelo cargado.")
        except Exception as e:
            logger.error(f"Error de carga: {e}")
            raise

    async def process_image(self, image_bytes: bytes, content_type: str) -> dict:
        from PIL import Image
        import io
        if content_type == "application/pdf":
            from pdf2image import convert_from_bytes
            page_images = convert_from_bytes(image_bytes, dpi=200)
        else:
            page_images = [Image.open(io.BytesIO(image_bytes)).convert("RGB")]

        all_system_lmx = []
        system_confidences = []
        total_systems = 0
        failed_systems = 0

        for page_idx, page_img in enumerate(page_images):
            n_systems = self._estimate_systems(page_img)
            system_images = self._segment_page(page_img, n_systems)
            for sys_idx, sys_img in enumerate(system_images):
                total_systems += 1
                try:
                    lmx_system, confidence = await asyncio.get_event_loop().run_in_executor(None, self._transcribe_system, sys_img)
                    all_system_lmx.append(lmx_system)
                    system_confidences.append(confidence)
                except Exception as e:
                    logger.error(f"Error sys {sys_idx + 1}: {e}")
                    failed_systems += 1

        lmx_complete = self._assemble_lmx(all_system_lmx)
        musicxml = self._lmx_to_musicxml(lmx_complete)
        success_rate = (total_systems - failed_systems) / max(total_systems, 1)
        structure_ok = "<?xml" in musicxml
        # Confianza real del modelo, media geometrica de la probabilidad que el propio modelo asigno a cada token que genero (log-prob por token, via compute_transition_scores)
        avg_confidence = sum(system_confidences) / len(system_confidences) if system_confidences else 0.0
        fiabilidad = round(success_rate * avg_confidence * (1.0 if structure_ok else 0.5), 3)

        return {"musicxml": musicxml, "lmx_raw": lmx_complete, "fiabilidad": fiabilidad, "metadatos": self._extract_metadata()}

    def _estimate_systems(self, pil_image) -> int:
        w, h = pil_image.size
        try:
            import cv2
            import numpy as np
            img_np = np.array(pil_image.convert("L"))
            h_img, w_img = img_np.shape
            _, binary = cv2.threshold(img_np, 200, 255, cv2.THRESH_BINARY_INV)
            kernel_len = max(w_img // 3, 50)
            kernel_h = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_len, 1))
            lines_img = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_h)
            projection = np.sum(lines_img, axis=1)
            threshold = w_img * 0.3
            line_rows = np.where(projection > threshold)[0]
            if len(line_rows) < 5: return 1
            bands, current = [], [line_rows[0]]
            for r in line_rows[1:]:
                if r - current[-1] <= 5: current.append(r)
                else:
                    bands.append(current)
                    current = [r]
            bands.append(current)
            n_estimated = max(1, round(len(bands) / 5))
            return min(n_estimated, max(1, h_img // 150), 8)
        except:
            return 1

    def _segment_page(self, pil_image, n_systems: int):
        try:
            import cv2
            result = self._segment_with_opencv(pil_image, n_systems)
            if result and len(result) == n_systems:
                return result
            logger.warning(f"Segmentacion OpenCV: {len(result) if result else 0} sistemas detectados (esperados {n_systems}); usando fallback proporcional.")
        except ImportError:
            logger.warning("OpenCV no disponible; usando fallback proporcional.")
        except Exception as e:
            logger.warning(f"Error en segmentacion OpenCV: {e}; usando fallback proporcional.")
        return self._segment_proportional(pil_image, n_systems)

    def _segment_with_opencv(self, pil_image, n_systems: int):
        """Detecta grupos de 5 lineas horizontales paralelas (pentagramas) y
        recorta cada sistema por sus coordenadas reales, igual que hace
        el proceso de preparación al construir el dataset de entrenamiento."""
        import cv2
        import numpy as np

        img_np = np.array(pil_image.convert("L"))
        h, w = img_np.shape
        _, binary = cv2.threshold(img_np, 200, 255, cv2.THRESH_BINARY_INV)
        kernel_len = max(w // 3, 50)
        kernel_h = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_len, 1))
        lines_img = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_h)
        projection = np.sum(lines_img, axis=1)
        threshold = w * 0.3
        line_rows = np.where(projection > threshold)[0]
        if len(line_rows) == 0:
            return None

        bands = []
        current_band = [line_rows[0]]
        for r in line_rows[1:]:
            if r - current_band[-1] <= 5:
                current_band.append(r)
            else:
                bands.append(current_band)
                current_band = [r]
        bands.append(current_band)

        if len(bands) < n_systems * 4:
            return None

        centers = [int(np.mean(b)) for b in bands]
        lines_per_system = max(1, len(centers) // n_systems)
        system_groups = [centers[i * lines_per_system:(i + 1) * lines_per_system] for i in range(n_systems)]

        margin = h // (n_systems * 4)
        crops = []
        for idx, group in enumerate(system_groups):
            if not group:
                continue
            top = max(0, group[0] - margin)
            if idx + 1 < len(system_groups) and system_groups[idx + 1]:
                bottom = min(h, system_groups[idx + 1][0] - margin // 2)
            else:
                bottom = h
            crops.append(pil_image.crop((0, top, w, bottom)))

        return crops if len(crops) == n_systems else None

    def _segment_proportional(self, pil_image, n_systems: int):
        w, h = pil_image.size
        slice_h = h // n_systems
        return [pil_image.crop((0, i * slice_h, w, (i + 1) * slice_h if i < n_systems - 1 else h)) for i in range(n_systems)]

    def _transcribe_system(self, system_image):
        from qwen_vl_utils import process_vision_info
        import torch
        messages = [{"role": "user", "content": [{"type": "image", "image": system_image}, {"type": "text", "text": SYSTEM_PROMPT_INFERENCE}]}]
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, _ = process_vision_info(messages)
        inputs = self.tokenizer(text=[text], images=image_inputs, padding=True, return_tensors="pt").to(self.model.device)
        with torch.inference_mode():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.MAX_NEW_TOKENS_SYSTEM,
                temperature=0.1,
                do_sample=False,
                output_scores=True,
                return_dict_in_generate=True,
            )
        generated_ids = outputs.sequences[:, inputs.input_ids.shape[1]:]
        generated = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        confidence = self._compute_confidence(outputs)
        return self._parse_lmx_from_output(generated), confidence

    def _compute_confidence(self, outputs) -> float:
        
        import torch
        try:
            transition_scores = self.model.compute_transition_scores(
                outputs.sequences, outputs.scores, normalize_logits=True
            )
            log_probs = transition_scores[0]
            log_probs = log_probs[torch.isfinite(log_probs)]
            if log_probs.numel() == 0:
                return 0.0
            return float(torch.exp(log_probs.mean()))
        except Exception as e:
            logger.warning(f"No se pudo calcular confianza por token: {e}")
            return 0.5

    def _parse_lmx_from_output(self, text: str) -> str:
        candidate = text.strip()
        for marker in ['```lmx', '```xml', '```']:
            if marker in candidate:
                parts = candidate.split(marker)
                if len(parts) >= 2:
                    candidate = parts[1].split("```")[0].strip()
                    break
        lines = [line for line in candidate.split("\n") if not re.match(r'^[XTMLKC]:', line)]
        candidate = "\n".join(lines).strip() if lines else candidate
        tokens = candidate.replace("\n", " ").split()
        if not candidate or candidate.startswith("<") or "measure" not in tokens:
            raise ValueError(
                "El modelo no devolvio LMX lineal valido. Comprueba que Deployment "
                "esta cargando el modelo fine-tuneado, no el modelo base."
            )
        return candidate

    def _assemble_lmx(self, system_lmxs: list) -> str:
        return " ".join([lmx.strip() for lmx in system_lmxs if lmx.strip()])

    def _lmx_to_musicxml(self, lmx_content: str) -> str:
        from lmx.linearization.Delinearizer import Delinearizer
        from lmx.symbolic.part_to_score import part_to_score
        import xml.etree.ElementTree as ET
        if not lmx_content.strip():
            raise ValueError("No se obtuvo ningun sistema LMX valido.")
        delinearizer = Delinearizer()
        delinearizer.process_text(lmx_content.replace('\n', ' '))
        score_etree = part_to_score(delinearizer.part_element)
        root = score_etree.getroot()
        measures = root.findall("./part/measure")
        if not measures:
            raise ValueError(
                "La conversion LMX produjo un MusicXML vacio, sin compases."
            )
        return str(
            ET.tostring(root, encoding='utf-8', xml_declaration=True),
            'utf-8',
        )

    def _extract_metadata(self) -> dict:
        return {"titulo": "Transcripción OMR LMX", "autor": "Qwen2.5-VL-OMR", "instrumento": "Piano", "genero": "Clásico", "anoOriginal": 2026}

In [ ]:
%%writefile /content/model_service.py

from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.responses import JSONResponse, StreamingResponse
from contextlib import asynccontextmanager
import asyncio
import json
import logging
import uvicorn

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

try:
    from omr_processor import OMRProcessor
    MODEL_AVAILABLE = True #pruebo false para comprobar que api funicona cuando no hay gpu
except ImportError:
    MODEL_AVAILABLE = False
    logger.warning("No se pudo cargar omr_processor, modo mock de inferencia activado")

omr_model = None

@asynccontextmanager
async def lifespan(app: FastAPI):
    global omr_model
    if MODEL_AVAILABLE:
        logger.info("Cargando modelo pesado Qwen2.5 en la GPU...")
        omr_model = OMRProcessor()
        logger.info("Modelo GPU listo para recibir peticiones internas.")
    else:
        logger.warning("Modelo GPU no disponible, correrá en modo mock interno.")
    yield

app = FastAPI(
    title="Worker Interno OMR GPU",
    description="Microservicio oculto para inferencia pesada",
    lifespan=lifespan
)

@app.get("/health-gpu")
async def health_check():
    return {
        "status": "ok",
        "gpu_model_loaded": omr_model is not None,
        "model_available": MODEL_AVAILABLE
    }

@app.post("/ai-predict")
async def ai_predict(file: UploadFile = File(...)):
    """
    Endpoint bloqueante interno que procesa la imagen a MusicXML
    """
    content = await file.read()

    try:
        if omr_model is not None:
            result = await omr_model.process_image(content, file.content_type)
        else:
            result = _mock_result()

        return JSONResponse(content=result)

    except Exception as e:
        logger.error(f"Error fatal internamente en la GPU: {e}", exc_info=True)
        raise HTTPException(status_code=500, detail=str(e))

async def _process_image_with_progress(content, content_type, report):
    """Mismo pipeline OMR, exponiendo los hitos reales al stream."""
    from PIL import Image
    import io

    await report({"type": "progress", "step": 0, "message": "Preparando la imagen para el reconocimiento..."})
    if content_type == "application/pdf":
        from pdf2image import convert_from_bytes
        page_images = convert_from_bytes(content, dpi=200)
    else:
        page_images = [Image.open(io.BytesIO(content)).convert("RGB")]

    await report({"type": "progress", "step": 1, "message": "Detectando los sistemas musicales..."})
    system_images = []
    for page_image in page_images:
        estimated = omr_model._estimate_systems(page_image)
        system_images.extend(omr_model._segment_page(page_image, estimated))

    total = len(system_images)
    lmx_systems = []
    confidences = []
    failed = 0
    loop = asyncio.get_running_loop()
    for current, system_image in enumerate(system_images, start=1):
        await report({
            "type": "progress",
            "step": 2,
            "message": f"Transcribiendo sistema {current} de {total} con la IA...",
            "current": current,
            "total": total,
        })
        try:
            lmx, confidence = await loop.run_in_executor(None, omr_model._transcribe_system, system_image)
            lmx_systems.append(lmx)
            confidences.append(confidence)
        except Exception as e:
            logger.error(f"Error sys {current}: {e}")
            failed += 1

    lmx_complete = omr_model._assemble_lmx(lmx_systems)
    await report({"type": "progress", "step": 3, "message": "Convirtiendo la transcripción a MusicXML..."})
    musicxml = await loop.run_in_executor(None, omr_model._lmx_to_musicxml, lmx_complete)
    await report({"type": "progress", "step": 4, "message": "Calculando el porcentaje de fiabilidad..."})
    success_rate = (total - failed) / max(total, 1)
    average_confidence = sum(confidences) / len(confidences) if confidences else 0.0
    reliability = round(success_rate * average_confidence * (1.0 if "<?xml" in musicxml else 0.5), 3)
    await report({"type": "progress", "step": 5, "message": "Transcripción completada. Preparando el resultado..."})
    return {
        "musicxml": musicxml,
        "lmx_raw": lmx_complete,
        "fiabilidad": reliability,
        "metadatos": omr_model._extract_metadata(),
    }

@app.post("/ai-predict-stream")
async def ai_predict_stream(file: UploadFile = File(...)):
    """Procesa una imagen y emite progreso real como NDJSON."""
    content = await file.read()

    async def events():
        queue = asyncio.Queue()

        async def report(event):
            await queue.put(event)

        async def run_inference():
            try:
                if omr_model is not None:
                    result = await _process_image_with_progress(content, file.content_type, report)
                else:
                    await report({"type": "progress", "step": 2, "message": "Ejecutando la transcripción de prueba..."})
                    result = _mock_result()
                await queue.put({"type": "result", "data": result})
            except Exception as e:
                logger.error(f"Error fatal internamente en la GPU: {e}", exc_info=True)
                await queue.put({"type": "error", "message": str(e)})
            finally:
                await queue.put(None)

        task = asyncio.create_task(run_inference())
        try:
            while True:
                event = await queue.get()
                if event is None:
                    break
                yield json.dumps(event, ensure_ascii=False) + "\n"
            await task
        finally:
            if not task.done():
                task.cancel()

    return StreamingResponse(
        events(),
        media_type="application/x-ndjson",
        headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},
    )

def _mock_result():
    import asyncio
    logger.info("Simulando inferencia...")

    mock_xml = """<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE score-partwise PUBLIC "-//Recordare//DTD MusicXML 4.0 Partwise//EN" "http://www.musicxml.org/dtds/partwise.dtd">
<score-partwise version="4.0">
  <work><work-title>Partitura Transcrita (Mock Backend GPU)</work-title></work>
  <identification><creator type="composer">Desconocido</creator></identification>
  <part-list><score-part id="P1"><part-name>Piano</part-name></score-part></part-list>
  <part id="P1">
    <measure number="1">
      <attributes>
        <divisions>1</divisions>
        <key><fifths>0</fifths></key>
        <time><beats>4</beats><beat-type>4</beat-type></time>
        <clef><sign>G</sign><line>2</line></clef>
      </attributes>
      <note><pitch><step>G</step><octave>4</octave></pitch><duration>1</duration><type>quarter</type></note>
      <note><pitch><step>E</step><octave>4</octave></pitch><duration>1</duration><type>quarter</type></note>
      <note><pitch><step>E</step><octave>4</octave></pitch><duration>2</duration><type>half</type></note>
    </measure>
  </part>
</score-partwise>"""
    return {
        "musicxml": mock_xml,
        "fiabilidad": 0.92,
        "metadatos": {
            "titulo": "Canción de Prueba (GPU)",
            "autor": "Desconocido",
            "instrumento": "Piano",
            "genero": "Clásico",
            "anoOriginal": 1800
        }
    }

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=8001)


In [ ]:
%%writefile /content/api.py


from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, StreamingResponse
from contextlib import asynccontextmanager
import uvicorn
import logging
from pathlib import Path
import base64
import os
from concurrent.futures import ThreadPoolExecutor
import asyncio
import json
import requests

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
_executor = ThreadPoolExecutor(max_workers=4)

@asynccontextmanager
async def lifespan(app: FastAPI):
    logger.info("Iniciando API Gateway PÚBLICA (Frontend -> Gateway -> GPU)")
    yield

app = FastAPI(
    title="OMR Multimodal API",
    description="API REST para transcripción de partituras manuscritas a MusicXML usando Qwen2.5",
    version="1.0.0",
    lifespan=lifespan,
)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  
    allow_methods=["POST", "GET", "OPTIONS"],
    allow_headers=["*"],
)

@app.get("/health")
async def health_check():
    """Verificación de estado de la API Gateway"""
    gpu_status = "unknown"
    try:
        r = requests.get("http://127.0.0.1:8001/health-gpu", timeout=2)
        if r.status_code == 200:
            gpu_status = "online"
    except Exception:
        gpu_status = "offline"

    return {
        "status": "ok",
        "role": "api_gateway",
        "gpu_worker": gpu_status
    }


@app.post("/transcribe")
async def transcribe_score(file: UploadFile = File(...)):
    ALLOWED_TYPES = ["image/jpeg", "image/png", "image/jpg", "application/pdf"]
    if file.content_type not in ALLOWED_TYPES:
        raise HTTPException(
            status_code=400,
            detail=f"Formato no admitido: {file.content_type}. Se aceptan JPEG, PNG y PDF."
        )

    content = await file.read()

    if len(content) > 10 * 1024 * 1024:
        raise HTTPException(status_code=400, detail="Imagen demasiado grande. Máximo 10 MB.")

    logger.info(f"Procesando imagen: {file.filename}, {len(content)} bytes")

    try:
        def _proxy_request():
            files = {'file': (file.filename, content, file.content_type)}
            response = requests.post("http://127.0.0.1:8001/ai-predict", files=files, timeout=300)
            if response.status_code == 200:
                return response.json()
            else:
                logger.error(f"Error desde el Worker GPU: {response.text}")
                raise Exception(f"Worker GPU reportó: {response.text}")

        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(_executor, _proxy_request)

        if result.get("fiabilidad", 0) < 0.85:
            logger.warning(f"Fiabilidad baja detectada en Gateway: {result.get('fiabilidad', 0):.1%}")

        return JSONResponse(content=result)

    except requests.exceptions.ConnectionError:
        logger.error("No se pudo conectar con el worker GPU en el puerto 8001")
        raise HTTPException(
            status_code=503,
            detail="Servicio de IA inactivo. El administrador debe arrancar model_service.py en el puerto 8001."
        )
    except Exception as e:
        logger.error(f"Error en transcripción gateway: {e}", exc_info=True)
        raise HTTPException(status_code=500, detail=f"Error al conectar con la GPU: {str(e)}")


@app.post("/transcribe-stream")
async def transcribe_score_stream(file: UploadFile = File(...)):
    """Proxy NDJSON que comunica al frontend el progreso real del worker."""
    allowed_types = ["image/jpeg", "image/png", "image/jpg", "application/pdf"]
    if file.content_type not in allowed_types:
        raise HTTPException(status_code=400, detail=f"Formato no admitido: {file.content_type}.")

    content = await file.read()
    if len(content) > 10 * 1024 * 1024:
        raise HTTPException(status_code=400, detail="Imagen demasiado grande. Máximo 10 MB.")

    async def events():
        loop = asyncio.get_running_loop()
        queue = asyncio.Queue()

        yield json.dumps({
            "type": "progress",
            "step": 0,
            "message": "Imagen recibida y validada. Enviando al procesador GPU...",
        }, ensure_ascii=False) + "\n"

        def proxy_worker_stream():
            try:
                files = {"file": (file.filename, content, file.content_type)}
                with requests.post(
                    "http://127.0.0.1:8001/ai-predict-stream",
                    files=files,
                    stream=True,
                    timeout=(10, 300),
                ) as response:
                    if response.status_code != 200:
                        raise RuntimeError(f"Worker GPU reportó: {response.text}")
                    response.encoding = "utf-8"
                    for line in response.iter_lines(decode_unicode=True, chunk_size=1):
                        if line:
                            loop.call_soon_threadsafe(queue.put_nowait, line)
            except Exception as e:
                logger.error(f"Error en el stream del worker GPU: {e}", exc_info=True)
                error_line = json.dumps({"type": "error", "message": str(e)}, ensure_ascii=False)
                loop.call_soon_threadsafe(queue.put_nowait, error_line)
            finally:
                loop.call_soon_threadsafe(queue.put_nowait, None)

        proxy_task = loop.run_in_executor(_executor, proxy_worker_stream)
        while True:
            line = await queue.get()
            if line is None:
                break
            yield line + "\n"
        await proxy_task

    return StreamingResponse(
        events(),
        media_type="application/x-ndjson",
        headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},
    )


if __name__ == "__main__":
    uvicorn.run(
        "api:app",
        host="0.0.0.0",
        port=8000,
        reload=True,
        reload_excludes=["unsloth_compiled_cache/*", "*.pyc"]
    )


## 4. Iniciar servicios
Lanza el Worker GPU interno (8001) y el API Gateway público (8000).

In [ ]:
import subprocess
import time

print('Iniciando Worker de GPU (Puerto 8001)...')
worker = subprocess.Popen(['python3', '/content/model_service.py'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

print('Iniciando API Gateway (Puerto 8000)...')
gateway = subprocess.Popen(['uvicorn', 'api:app', '--host', '0.0.0.0', '--port', '8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

time.sleep(5)
print('Servicios lanzados en segundo plano.')

## 5. Abrir túnel de Cloudflare
Copia la URL `https://...trycloudflare.com` y pégala en el archivo `.env` local (`VITE_OMR_API_URL`).

In [ ]:
import subprocess
import threading
import time

print("Iniciando túnel Cloudflare...")
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

def print_tunnel_output():
    for line in tunnel.stdout:
        print(f"[Cloudflared] {line.strip()}")
        if "trycloudflare.com" in line and "https://" in line:
            # Extracción segura
            url = next((w for w in line.split() if w.startswith("https://")), None)
            if url:
                print("\n" + "="*60 + f"\n👉 URL PUBLICA: {url}\n" + "="*60 + "\n")
            break

threading.Thread(target=print_tunnel_output, daemon=True).start()

def print_worker_output():
    for line in worker.stdout:
        print(f"[GPU Worker] {line.decode('utf-8').strip()}")

threading.Thread(target=print_worker_output, daemon=True).start()

try:
    while True:
        line = gateway.stdout.readline()
        if line:
            print(f"[API Gateway] {line.decode('utf-8').strip()}")
        else:
            time.sleep(0.1)
except KeyboardInterrupt:
    print("Deteniendo...")
    tunnel.terminate()
    worker.terminate()
    gateway.terminate()
